# Validation 1: Sarvam Document Intelligence — OCR

**Objective:** Evaluate Sarvam **Document Intelligence** on custom images and PDFs for OCR fidelity, layout understanding, visual reasoning on charts, and Indic/multilingual robustness — validated by numeric accuracy and human review.

1. Run cells **3 → 7** once (setup → OCR pipeline → Gradio viewer)
2. Run any image/PDF section below

**Output:** Gradio panel with **input document** (left) and **OCR markdown** (right).<br><br>
**Validation axes:**<br>
a. OCR fidelity · b. Layout understanding · c. Visual reasoning · d. Multilingual + Indic robustness<br><br>
All inputs under `test_inputs/images to test for text extraction quality/` and `test_inputs/pdfs to test for text extraction quality/`.

## Summary of results: <br>

| Sr no  | Image/PDF   | Description of input | What it tests? | OCR outcome   | Output | Comments
|---|---|---|---|---| --- | --- |
|  1 | Image (.png)  | Bar chart with minimal text  | axis extraction, label association, rotated labels, numeric extraction from axis, charts and legends  | Failed | BadrequestError: message': 'Invalid or corrupted image file.' | Challenges in handling extraction from comple charts |
| 2  | Image  | 2-dimensional time series, dense bar chart with labels, annotations and legends   | stacked bar interpretation, category grouping, color-legend alignment, numeric grounding   | Failed | BadrequestError: message': 'Invalid or corrupted image file.' | Challenges in handling extraction from comple charts |
| 3  | Image  | Rotated annotated 2-dim line chart | orientation robustness, tilted axis reading, layout recovery   | Success  | Complete reconstruction + layout understanding | |
| 4  | Image   | Pure hindi newspaper scanned | Hindi OCR, multicolumn reading order, headline detection, semantic layout segmentation   | Success  | 100% text accuracy and layout understanding | 
| 5  | Image  | Clean table   | row-column understanding, structured extraction, numeric precision  |  Success  | 100% numerical accuracy and layout understanding |
| 6  | Image   | Real-world degraded + stamped table scanned| noisy scan robustness, broken lines, merged cells, low DPI OCR  | Partial Success  | Extraction poor where stamp and degradation is present | |
| 7  | Image   | Code-Mixed + handwritten form  | handwritten OCR, Hinglish/code-switching, key-value extraction, form layout understanding  | Success | Handles extraction of table of both handwritten and hindi scripts | |
| 8  | Image  | Pseduo-code with mathematical equation   | equation parsing, symbol recognition, superscripts/subscripts, LaTeX reconstruction  | Partial Success  | ~80% character level accuracy | Challenges in : <br>(a) Mathematical superscripts/subscripts <br>(b) Greek symbols <br>(c) loops and indentation flattened and <br>(d) Special symbols such as Transpose T
| 9  | PDF   | 11 pages english document   | Stress test for edge-case >10 pages document   | Correct Error Handling  | The error message accurately pointed > 10 pages | | |
| 10  | PDF   | 1 page Highlighted text pdf| OCR quality from pdfs with highlights | Success  | 100% text extraction accuracy despite the highlighted text | |


## Setup — paths, API key, input folders

In [ ]:
import json
import os
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
import gradio as gr
from sarvamai import SarvamAI

ROOT = Path.cwd()
load_dotenv(ROOT / ".env")

# PNG/JPEG charts, tables, forms — under test_inputs/
IMAGE_DIR = ROOT / "test_inputs/images to test for text extraction quality"
# PDFs for multi-page and highlight tests
DOC_DIR = ROOT / "test_inputs/pdfs to test for text extraction quality"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

_client: SarvamAI | None = None


def get_sarvam_client() -> SarvamAI:
    """Reuse one Document Intelligence client per session."""
    global _client
    if _client is None:
        api_key = os.getenv("SARVAM_API_KEY")
        if not api_key:
            raise ValueError("SARVAM_API_KEY not set — add it to .env")
        _client = SarvamAI(api_subscription_key=api_key)
    return _client


print("SARVAM_API_KEY:", "set" if os.getenv("SARVAM_API_KEY") else "MISSING")
print("Images:", sorted(p.name for p in IMAGE_DIR.glob("*.png")))
print("Documents:", sorted(p.name for p in DOC_DIR.glob("*.pdf")))


## Document Intelligence — job runner + `OcrResult`

In [ ]:
@dataclass
class OcrResult:
    """Sarvam Document Intelligence output for one image or PDF."""

    document_path: Path
    success: bool
    runtime_total_seconds: float
    output_dir: Path | None = None
    markdown: str = ""
    job_state: str | None = None
    pages_processed: int = 0
    pages_failed: int = 0
    metadata: list[dict[str, Any]] | None = None
    error_type: str | None = None
    error_message: str | None = None
    error_code: str | int | None = None

    @property
    def name(self) -> str:
        return self.document_path.name

    @property
    def is_pdf(self) -> bool:
        return self.document_path.suffix.lower() == ".pdf"


def _extract_error_info(exc: Exception) -> tuple[str, str, str | int | None]:
    from sarvamai.core.api_error import ApiError

    error_type = type(exc).__name__
    error_message = str(exc)
    error_code: str | int | None = None
    if isinstance(exc, ApiError):
        error_code = exc.status_code
        if exc.body is not None:
            error_message = str(exc.body)
    elif getattr(exc, "status_code", None) is not None:
        error_code = getattr(exc, "status_code")
    return error_type, error_message, error_code


def _print_error(document_name: str, runtime_total: float, exc: Exception):
    error_type, error_message, error_code = _extract_error_info(exc)
    print(f"✗ {document_name} failed")
    print(f"  Runtime total : {runtime_total}s")
    print(f"  Error type    : {error_type}")
    print(f"  Error code    : {error_code if error_code is not None else 'N/A'}")
    print(f"  Error message : {error_message}")
    return error_type, error_message, error_code


def _read_sarvam_outputs(extract_dir: Path) -> tuple[str, list[dict[str, Any]]]:
    """Pull .md and metadata/*.json from the extracted zip."""
    md_files = sorted(extract_dir.rglob("*.md"))
    markdown = md_files[0].read_text(encoding="utf-8") if md_files else ""
    metadata = [
        json.loads(p.read_text(encoding="utf-8"))
        for p in sorted(extract_dir.rglob("metadata/*.json"))
    ]
    return markdown, metadata


def run_sarvam_ocr(
    document_path: str | Path,
    *,
    language: str = "en-IN",
    output_format: str = "md",
    output_root: Path | str = RESULTS_DIR,
) -> OcrResult:
    """Upload → job → wait → download zip → read markdown + metadata."""
    document_path = Path(document_path)
    started = time.perf_counter()

    if not document_path.exists():
        exc = FileNotFoundError(document_path)
        runtime_total = round(time.perf_counter() - started, 2)
        err_type, err_msg, err_code = _print_error(document_path.name, runtime_total, exc)
        return OcrResult(
            document_path=document_path,
            success=False,
            runtime_total_seconds=runtime_total,
            error_type=err_type,
            error_message=err_msg,
            error_code=err_code,
        )

    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)
    extract_dir = output_root / document_path.stem
    output_zip = output_root / f"{document_path.stem}.zip"

    try:
        client = get_sarvam_client()
        job = client.document_intelligence.create_job(
            language=language,
            output_format=output_format,
        )
        job.upload_file(str(document_path))
        job.start()
        status = job.wait_until_complete()
        metrics = job.get_page_metrics()
        job.download_output(str(output_zip))

        with zipfile.ZipFile(output_zip, "r") as zf:
            zf.extractall(extract_dir)

        markdown, metadata = _read_sarvam_outputs(extract_dir)
        runtime_total = round(time.perf_counter() - started, 2)

        print(f"✓ {document_path.name} completed")
        print(f"  Runtime total : {runtime_total}s")
        print(f"  Job state     : {status.job_state}")
        print(
            f"  Pages OK/Fail : {metrics.get('pages_processed', 0)}/"
            f"{metrics.get('pages_failed', 0)}"
        )

        return OcrResult(
            document_path=document_path.resolve(),
            success=True,
            runtime_total_seconds=runtime_total,
            output_dir=extract_dir.resolve(),
            markdown=markdown,
            job_state=status.job_state,
            pages_processed=metrics.get("pages_processed", 0),
            pages_failed=metrics.get("pages_failed", 0),
            metadata=metadata,
        )

    except Exception as exc:
        runtime_total = round(time.perf_counter() - started, 2)
        err_type, err_msg, err_code = _print_error(document_path.name, runtime_total, exc)
        return OcrResult(
            document_path=document_path.resolve(),
            success=False,
            runtime_total_seconds=runtime_total,
            error_type=err_type,
            error_message=err_msg,
            error_code=err_code,
        )


## Gradio viewer — document preview + OCR markdown

In [ ]:
def _format_status(result: OcrResult) -> str:
    if not result.success:
        return (
            f"**Failed** · {result.runtime_total_seconds}s · "
            f"code `{result.error_code}` · {result.error_type}\n\n"
            f"{result.error_message}"
        )
    return (
        f"**{result.job_state}** · {result.runtime_total_seconds}s · "
        f"{result.pages_processed} page(s) ok"
    )


def _format_ocr_text(result: OcrResult) -> str:
    if not result.success:
        return result.error_message or "Unknown error"
    return result.markdown or "(no text extracted)"


def show_ocr_result(result: OcrResult, *, window_height: int = 950) -> gr.Blocks:
    """Gradio: image or PDF file (left) | OCR markdown (right)."""
    input_path = str(result.document_path.resolve())
    ocr_text = _format_ocr_text(result)
    is_pdf = bool(getattr(result, "is_pdf", str(result.document_path).lower().endswith(".pdf")))

    with gr.Blocks(title=result.name) as demo:
        gr.Markdown(f"### {result.name}\n\n{_format_status(result)}")
        with gr.Row(equal_height=True):
            if is_pdf:
                gr.File(
                    value=input_path,
                    label="Input PDF",
                    interactive=False,
                )
            else:
                gr.Image(
                    value=input_path,
                    label="Input Image",
                    height=420,
                    interactive=False,
                )
            gr.Markdown(
                value=ocr_text,
                label="OCR output",
            )

    demo.launch(
        inline=True,
        height=window_height,
        share=False,
        quiet=True,
        allowed_paths=[str(ROOT), str(IMAGE_DIR), str(DOC_DIR), str(RESULTS_DIR)],
    )
    return demo


## Image & PDF runs

Run one cell at a time; fill the summary table above after review.

## Images — charts (simple & dense)

**plot1** / **plot4** — bar charts; API returned invalid/corrupted image on these two in our run.

In [ ]:
result_plot1 = run_sarvam_ocr(IMAGE_DIR / "plot1_atc_feeders.png")
show_ocr_result(result_plot1)

✗ plot1_atc_feeders.png failed
  Runtime total : 0.83s
  Error type    : BadRequestError
  Error code    : 400
  Error message : {'error': {'message': 'Invalid or corrupted image file.', 'code': 'invalid_request_error', 'request_id': '20260520_082da048-339b-4306-a07e-47dc86304900'}}


Gradio Blocks instance: 0 backend functions
-------------------------------------------

In [ ]:
result_plot2 = run_sarvam_ocr(IMAGE_DIR / "plot4_pending_arrears.png")
show_ocr_result(result_plot2)

✗ plot4_pending_arrears.png failed
  Runtime total : 2.38s
  Error type    : BadRequestError
  Error code    : 400
  Error message : {'error': {'message': 'Invalid or corrupted image file.', 'code': 'invalid_request_error', 'request_id': '20260520_84c2c720-b378-4dfb-82ce-32652d27f5c0'}}


Gradio Blocks instance: 0 backend functions
-------------------------------------------

## Image — rotated line chart (plot5)

Orientation robustness, tilted axis labels.

In [ ]:
result_plot5 = run_sarvam_ocr(IMAGE_DIR / "plot5_acos_solarisation_rotated.jpeg")
show_ocr_result(result_plot5)

✓ plot5_acos_solarisation_rotated.jpeg completed
  Runtime total : 8.11s
  Job state     : Completed
  Pages OK/Fail : 1/0


Gradio Blocks instance: 0 backend functions
-------------------------------------------

## Image — pure Hindi newspaper scan (plot7)

Multicolumn Hindi OCR + layout segmentation.

In [ ]:
result_plot6 = run_sarvam_ocr(IMAGE_DIR / "plot7_pure_hindi.png")
show_ocr_result(result_plot6)

✓ plot7_pure_hindi.png completed
  Runtime total : 12.45s
  Job state     : Completed
  Pages OK/Fail : 1/0


Gradio Blocks instance: 0 backend functions
-------------------------------------------

## Image — clean table (plot8)

Row/column structure and numeric precision.

In [ ]:
result_plot7 = run_sarvam_ocr(IMAGE_DIR / "plot8_table.png")
show_ocr_result(result_plot7)

✓ plot8_table.png completed
  Runtime total : 7.44s
  Job state     : Completed
  Pages OK/Fail : 1/0


Gradio Blocks instance: 0 backend functions
-------------------------------------------

## Image — degraded stamped table (plot9)

Noisy scan; matplotlib preview because Gradio is cramped for dense tables.

In [ ]:
result_plot8 = run_sarvam_ocr(IMAGE_DIR / "plot9_scanned_table_poor_qlty.png")
# show_ocr_result(result_plot8)  # Gradio not ideal for very dense table output

import matplotlib.pyplot as plt
from PIL import Image as PILImage

img = PILImage.open(IMAGE_DIR / "plot9_scanned_table_poor_qlty.png")
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis("off")
plt.show()
result_plot8.markdown[:500] if result_plot8.success else result_plot8.error_message

## Image — Hinglish handwritten form (plot10)

In [ ]:
result_plot9 = run_sarvam_ocr(IMAGE_DIR / "plot10_hinglish_form.png")
show_ocr_result(result_plot9)

✓ plot10_hinglish_form.png completed
  Runtime total : 7.48s
  Job state     : Completed
  Pages OK/Fail : 1/0


Gradio Blocks instance: 0 backend functions
-------------------------------------------

## Image — pseudo-code + math (plot11)

Superscripts, Greek symbols, indentation.

In [ ]:
result_plot10 = run_sarvam_ocr(IMAGE_DIR / "plot11_math_equations.png")
show_ocr_result(result_plot10)

✓ plot11_math_equations.png completed
  Runtime total : 7.38s
  Job state     : Completed
  Pages OK/Fail : 1/0


Gradio Blocks instance: 0 backend functions
-------------------------------------------

## PDF — highlighted text, 1 page (doc2)

In [ ]:
result_pdf = run_sarvam_ocr(DOC_DIR / "doc2_highlighted_text.pdf")
show_ocr_result(result_pdf)

✓ doc2_highlighted_text.pdf completed
  Runtime total : 7.32s
  Job state     : Completed
  Pages OK/Fail : 1/0


Gradio Blocks instance: 0 backend functions
-------------------------------------------

## PDF — 11 pages edge case (doc1)

Expect rejection when >10 pages.

In [ ]:
result_pdf2 = run_sarvam_ocr(DOC_DIR / "doc1_11_pages_edge.pdf")
show_ocr_result(result_pdf2)

✗ doc1_11_pages_edge.pdf failed
  Runtime total : 1.6s
  Error type    : BadRequestError
  Error code    : 400
  Error message : {'error': {'message': 'PDF has 11 pages, maximum allowed is 10.', 'code': 'invalid_request_error', 'request_id': '20260520_ad8d23d6-65cc-42df-b286-2a2e5b56e704'}}


Gradio Blocks instance: 0 backend functions
-------------------------------------------

## PDF — dual-column English (doc3)

In [ ]:
result_pdf3 = run_sarvam_ocr(DOC_DIR / "doc3_pure_english_dual_column.pdf")
show_ocr_result(result_pdf3)

✓ doc3_pure_english_dual_column.pdf completed
  Runtime total : 8.3s
  Job state     : Completed
  Pages OK/Fail : 1/0


Gradio Blocks instance: 0 backend functions
-------------------------------------------